---
# Part II – Tweets Emotion Classification using Word Embeddings

In [1]:
import pandas as pd
import numpy as np
import gensim.downloader as gensim_api
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [2]:
twdataset = pd.read_csv('twitter_emotion.csv')
print(twdataset.shape)
print(twdataset.head())

(416809, 3)
   Unnamed: 0                                               text  label
0           0      i just feel really helpless and heavy hearted      4
1           1  ive enjoyed being able to slouch about relax a...      0
2           2  i gave up my internship with the dmrg and am f...      4
3           3                         i dont know i feel so lost      0
4           4  i am a kindergarten teacher and i am thoroughl...      4


In [3]:
TWTEXTCOL  = twdataset.columns[0] 
TWLABELCOL = twdataset.columns[-1] 
print(f'Text column : {TWTEXTCOL}')
print(f'Label column: {TWLABELCOL}')
print('\nEmotion distribution:')
print(twdataset[TWLABELCOL].value_counts())

Text column : Unnamed: 0
Label column: label

Emotion distribution:
label
1    141067
0    121187
3     57317
4     47712
2     34554
5     14972
Name: count, dtype: int64


## 1. Tweets Pre-processing – Keras Tokenizer

In [4]:
MAX_WORDS   = 100_000
MAX_SEQ_LEN = 500          # default; the experiment loop will override this

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(twdataset[TWTEXTCOL].astype(str))
sequences  = tokenizer.texts_to_sequences(twdataset[TWTEXTCOL].astype(str))
word_index = tokenizer.word_index

print(f'Vocab size        : {len(word_index)}')
print(f'First 10 entries  : {list(word_index.items())[:10]}')

padded_sequences = pad_sequences(
    sequences,
    maxlen=MAX_SEQ_LEN,
    padding='post',
    truncating='post'
)
print(f'Padded sequences shape: {padded_sequences.shape}')

# Quick stats to help choose MAX_LEN values
raw_lengths = [len(seq) for seq in sequences]
print(f'\nTweet length stats:')
print(f'  Mean   : {np.mean(raw_lengths):.0f} tokens')
print(f'  Median : {np.median(raw_lengths):.0f} tokens')
print(f'  95th % : {np.percentile(raw_lengths, 95):.0f} tokens')
print(f'  99th % : {np.percentile(raw_lengths, 99):.0f} tokens')
print(f'  Max    : {max(raw_lengths)} tokens')

Vocab size        : 416810
First 10 entries  : [('<OOV>', 1), ('0', 2), ('1', 3), ('2', 4), ('3', 5), ('4', 6), ('5', 7), ('6', 8), ('7', 9), ('8', 10)]
Padded sequences shape: (416809, 500)

Tweet length stats:
  Mean   : 1 tokens
  Median : 1 tokens
  95th % : 1 tokens
  99th % : 1 tokens
  Max    : 1 tokens


## 2a. Pre-trained Embeddings – GloVe Twitter 50d

In [5]:
glove_model = gensim_api.load('glove-twitter-50')
EMBEDDING_DIM_GLOVE = 50
print(f'Vocabulary size in GloVe model: {len(glove_model.key_to_index)}')

Vocabulary size in GloVe model: 1193514


In [6]:
def buildEmbeddingMatrix(wvModel, word_index, max_words, embedding_dimension):
    matrix = np.zeros((max_words + 1, embedding_dimension))
    for word, idx in word_index.items():
        if idx > max_words:
            continue
        try:
            matrix[idx] = wvModel[word]
        except KeyError:
            pass   
    return matrix
glove_embedding_matrix = buildEmbeddingMatrix(
    glove_model, word_index, MAX_WORDS, EMBEDDING_DIM_GLOVE
)
print(f'GloVe embedding matrix shape: {glove_embedding_matrix.shape}')

GloVe embedding matrix shape: (100001, 50)


## 2b. Your trained word2vec model word embeddings:

In [7]:
print("Step 2b – Training custom Word2Vec model …")

indx_to_word = {idx: word for word, idx in word_index.items()}
tokenized_tweets = [
    [indx_to_word[idx] for idx in seq if idx in indx_to_word]
    for seq in sequences
]

print(f"Total tokenized tweets : {len(tokenized_tweets)}")
print(f"First tokenized tweet  : {tokenized_tweets[0]}")

w2v_model = Word2Vec(
    sentences=tokenized_tweets,
    vector_size=100,
    window=5,
    min_count=1,
    epochs=10,
    sg=1
)

w2v_model.save("word2vec_twitter.model")
print(f"Word2Vec size : {len(w2v_model.wv.key_to_index)}")

print("Building Word2Vec embedding matrix")
w2v_embedding_matrix = buildEmbeddingMatrix(
    w2v_model.wv, word_index, MAX_WORDS, 100
)
print(f"Word2Vec embedding matrix shape: {w2v_embedding_matrix.shape}")

Step 2b – Training custom Word2Vec model …
Total tokenized tweets : 416809
First tokenized tweet  : ['0']
Word2Vec size : 99999
Building Word2Vec embedding matrix
Word2Vec embedding matrix shape: (100001, 100)


## 3. Data preparation and splitting:

3a. Apply one-hot encoding for the integer labels using keras.

In [8]:
from tensorflow.keras.utils import to_categorical

labels = twdataset[TWLABELCOL].values
one_hot_labels = to_categorical(labels)

print("Original labels shape:", labels.shape)
print("One-hot labels shape:", one_hot_labels.shape)

Original labels shape: (416809,)
One-hot labels shape: (416809, 6)


3b. Splitting


In [9]:
from sklearn.model_selection import train_test_split

# split 80% & 20%

X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
    padded_sequences,       # input features
    one_hot_labels,         # labels
    test_size=0.2,          # 20% test
    random_state=42,        
    shuffle=True
)

print("\n80/20 Split:")
print("X_train shape:", X_train_80.shape)
print("X_test shape :", X_test_80.shape)


# split 70% & 30%

X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(
    padded_sequences,
    one_hot_labels,
    test_size=0.3,          
    random_state=42,
    shuffle=True
)

print("\n70/30 Split:")
print("X_train shape:", X_train_70.shape)
print("X_test shape :", X_test_70.shape)


80/20 Split:
X_train shape: (333447, 500)
X_test shape : (83362, 500)

70/30 Split:
X_train shape: (291766, 500)
X_test shape : (125043, 500)


4.1 — Model Builder Function

In [13]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Embedding, Conv1D, MaxPooling1D, Flatten, Dense, Input
)
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

NUM_CLASSES = one_hot_labels.shape[1]       # 6

def build_cnn_model(embedding_matrix, embedding_dim, max_len, name="CNN"):  # ← added max_len
    vocab_size = embedding_matrix.shape[0]

    # Input layer
    sequence_input = Input(shape=(max_len,), dtype='int32')                  # ← max_len not MAX_LEN
    embedding_layer = Embedding(
        vocab_size,                   # ← FIX: was len(vocab_size) which crashes
        embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,         # ← max_len not MAX_LEN
        trainable=True
    )

    # Embedding layer
    embedded_sequences = embedding_layer(sequence_input)

    # 1st Conv1D layer
    Conv_Layer1 = Conv1D(128, 5, activation='relu')(embedded_sequences)
    MxPooling_Layer1 = MaxPooling1D(5)(Conv_Layer1)

    # 2nd Conv1D layer
    Conv_Layer2 = Conv1D(128, 5, activation='relu')(MxPooling_Layer1)
    MxPooling_Layer2 = MaxPooling1D(5)(Conv_Layer2)

    # Output layer
    Flatten_layer = Flatten()(MxPooling_Layer2)          # ← FIX: was MaxPooling_Layer2 (typo)
    Dense_layer = Dense(128, activation='relu')(Flatten_layer)
    output_layer = Dense(NUM_CLASSES, activation='softmax')(Dense_layer)  # ← FIX: was n_labels

    model = Model(sequence_input, output_layer)

    model.compile(
        loss='categorical_crossentropy',
        optimizer='rmsprop',
        metrics=['accuracy']
    )

    return model

4.2 — Training & Plotting Helpers

In [15]:
def train_and_evaluate(model, X_train, X_test, y_train, y_test,
                       epochs=15, batch_size=128):

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\n★ Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.4f}\n")
    return history, test_loss, test_acc


def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.plot(history.history['accuracy'],    label='Train Acc')
    ax1.plot(history.history['val_accuracy'], label='Val Acc')
    ax1.set_title(f'{title} — Accuracy')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(); ax1.grid(True)

    ax2.plot(history.history['loss'],    label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title(f'{title} — Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(True)

    plt.tight_layout(); plt.show()

4.3 — Run All 4 Experiments

In [16]:
import pandas as pd
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

for MAX_LEN in [500, 700, 1000]:

    print("\n" + "█" * 70)
    print(f"  MAX_LEN = {MAX_LEN}")
    print("█" * 70)

    padded = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

    X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
        padded, one_hot_labels, test_size=0.2, random_state=42, shuffle=True
    )
    X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(
        padded, one_hot_labels, test_size=0.3, random_state=42, shuffle=True
    )

    results = {}

    # ── GloVe + 80/20 ──
    print("=" * 60)
    print(f"  GloVe-50d  |  80/20 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    model_glove_80 = build_cnn_model(glove_embedding_matrix, EMBEDDING_DIM_GLOVE, MAX_LEN, name="CNN_GloVe_80_20")
    model_glove_80.summary()
    hist, loss, acc = train_and_evaluate(model_glove_80, X_train_80, X_test_80, y_train_80, y_test_80)
    plot_history(hist, f"GloVe-50d | 80/20 | MAX_LEN={MAX_LEN}")
    results["GloVe-50d | 80/20"] = {"loss": loss, "accuracy": acc}

    # ── GloVe + 70/30 ──
    print("=" * 60)
    print(f"  GloVe-50d  |  70/30 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    model_glove_70 = build_cnn_model(glove_embedding_matrix, EMBEDDING_DIM_GLOVE, MAX_LEN, name="CNN_GloVe_70_30")
    hist, loss, acc = train_and_evaluate(model_glove_70, X_train_70, X_test_70, y_train_70, y_test_70)
    plot_history(hist, f"GloVe-50d | 70/30 | MAX_LEN={MAX_LEN}")
    results["GloVe-50d | 70/30"] = {"loss": loss, "accuracy": acc}

    # ── Word2Vec + 80/20 ──
    print("=" * 60)
    print(f"  Word2Vec-100d  |  80/20 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    model_w2v_80 = build_cnn_model(w2v_embedding_matrix, EMBEDDING_DIM_W2V, MAX_LEN, name="CNN_W2V_80_20")
    model_w2v_80.summary()
    hist, loss, acc = train_and_evaluate(model_w2v_80, X_train_80, X_test_80, y_train_80, y_test_80)
    plot_history(hist, f"Word2Vec-100d | 80/20 | MAX_LEN={MAX_LEN}")
    results["Word2Vec-100d | 80/20"] = {"loss": loss, "accuracy": acc}

    # ── Word2Vec + 70/30 ──
    print("=" * 60)
    print(f"  Word2Vec-100d  |  70/30 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    model_w2v_70 = build_cnn_model(w2v_embedding_matrix, EMBEDDING_DIM_W2V, MAX_LEN, name="CNN_W2V_70_30")
    hist, loss, acc = train_and_evaluate(model_w2v_70, X_train_70, X_test_70, y_train_70, y_test_70)
    plot_history(hist, f"Word2Vec-100d | 70/30 | MAX_LEN={MAX_LEN}")
    results["Word2Vec-100d | 70/30"] = {"loss": loss, "accuracy": acc}

    # ── Save this run's results ──
    results_df = pd.DataFrame(results).T
    results_df.index.name = "Experiment"
    results_df = results_df.round(4)
    results_df.to_csv(f"results_maxlen_{MAX_LEN}.csv")
    print(f"\n💾 Saved: results_maxlen_{MAX_LEN}.csv")
    print(results_df.to_string())


██████████████████████████████████████████████████████████████████████
  MAX_LEN = 500
██████████████████████████████████████████████████████████████████████
  GloVe-50d  |  80/20 Split  |  MAX_LEN=500


f:\senior stage\2nd sem\NLP\Text_Classification_NLP_Project\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 500)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 500, 50)        │     5,000,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 496, 128)       │        32,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 99, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 95, 128)        │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 19, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2432)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       311,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,426,424 (20.70 MB)

 Trainable params: 5,426,424 (20.70 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
 517/2345 ━━━━━━━━━━━━━━━━━━━━ 3:03 100ms/step - accuracy: 0.3330 - loss: 1.7278

KeyboardInterrupt: 

4.4 — Results Comparison Table

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).T
results_df.index.name = "Experiment"
results_df = results_df.round(4)
print("\n")
print(results_df.to_string())



                         loss  accuracy
Experiment                             
GloVe-50d | 80/20      1.5745    0.3379
GloVe-50d | 70/30      1.5745    0.3379
Word2Vec-100d | 80/20  1.5756    0.3379
Word2Vec-100d | 70/30  1.5781    0.3379
